In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import wandb

# WandB login
wandb.login(key='0530bbf36999cf23c9faffc230ca42a929ee045b')
entity = 'cs24s023-iitm-ac-in'
project = 'garbage_classification'

# Dataset setup
DATA_PATH = '/home/nagaraj/Garbage_classification_files/Garbage classification'
CLASSES = ['cardboard','glass','metal','paper','plastic','trash']

transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(DATA_PATH, transform=transform)


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/nagaraj/.netrc
wandb: Currently logged in as: cs24s023 (cs24s023-iitm-ac-in) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
DATA_PATH = '/home/nagaraj/Garbage_classification_files/Garbage classification'
CLASSES = ['cardboard','glass','metal','paper','plastic','trash']

transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(DATA_PATH, transform=transform)

# Split dataset: 80% train, 10% val, 10% test
train_len = int(0.8 * len(dataset))
val_test_len = len(dataset) - train_len
train_data, val_test_data = random_split(dataset, [train_len, val_test_len])
val_len = test_len = val_test_len // 2
val_data, test_data = random_split(val_test_data, [val_len, test_len])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)


In [3]:
class GarbageCNN(nn.Module):
    def __init__(self, filters=[32,64,128,128,256], kernel=3, activation='relu',
                 dense_units=512, dropout=0.3, use_bn=True):
        super().__init__()
        self.layers = nn.ModuleList()
        in_ch = 3
        self.use_bn = use_bn
        self.activation = activation
        
        for out_ch in filters:
            self.layers.append(nn.Conv2d(in_ch, out_ch, kernel_size=kernel, padding=1))
            if use_bn:
                self.layers.append(nn.BatchNorm2d(out_ch))
            self.layers.append(nn.MaxPool2d(2,2))
            in_ch = out_ch

        self.flatten = nn.Flatten()
        sample_input = torch.zeros(1,3,128,128)
        for layer in self.layers:
            sample_input = layer(sample_input)
        fc_input_features = sample_input.numel()
        self.fc1 = nn.Linear(fc_input_features, dense_units)
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(dense_units, len(CLASSES))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
            if isinstance(layer, nn.Conv2d) or isinstance(layer, nn.BatchNorm2d):
                if self.activation=='relu': x = F.relu(x)
                elif self.activation=='gelu': x = F.gelu(x)
                elif self.activation=='silu': x = F.silu(x)
                elif self.activation=='mish': x = x*torch.tanh(F.softplus(x))
        x = self.flatten(x)
        x_fc = self.fc1(x)
        if self.activation=='relu': x_fc = F.relu(x_fc)
        elif self.activation=='gelu': x_fc = F.gelu(x_fc)
        elif self.activation=='silu': x_fc = F.silu(x_fc)
        elif self.activation=='mish': x_fc = x_fc*torch.tanh(F.softplus(x_fc))
        x_fc = self.drop(x_fc)
        return self.fc2(x_fc)


In [4]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def estimate_flops(model, input_size=(1,3,128,128)):
    flops = 0
    x = torch.zeros(input_size)
    for layer in model.layers:
        if isinstance(layer, nn.Conv2d):
            out_ch, in_ch, k, _ = layer.weight.shape
            h,w = x.shape[2], x.shape[3]
            flops += 2*h*w*in_ch*out_ch*k*k
        x = layer(x)
    in_f = model.fc1.in_features
    out_f = model.fc1.out_features
    flops += 2*in_f*out_f
    flops += 2*out_f*model.fc2.out_features
    return flops


In [5]:
def train_sweep():
    wandb.init()
    cfg = wandb.config
    model = GarbageCNN(filters=cfg.filters, activation=cfg.activation,
                       dropout=cfg.dropout, use_bn=cfg.batch_norm,
                       dense_units=cfg.dense_neurons).to(device)

    print(f"Trainable Parameters: {count_parameters(model):,}")
    print(f"Estimated FLOPs: {estimate_flops(model):,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(10):
        model.train()
        total_loss, correct, total = 0,0,0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            _, pred = out.max(1)
            correct += (pred==labels).sum().item()
            total += labels.size(0)
        train_acc = correct/total

        # Validation
        model.eval()
        correct_val, total_val = 0,0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                out = model(imgs)
                _, pred = out.max(1)
                correct_val += (pred==labels).sum().item()
                total_val += labels.size(0)
        val_acc = correct_val/total_val
        wandb.log({'train_loss': total_loss/len(train_loader), 
                   'train_acc': train_acc, 'val_acc': val_acc})

    # Test Evaluation
    model.eval()
    correct_test, total_test = 0,0
    all_imgs, all_preds, all_labels = [],[],[]
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            _, pred = out.max(1)
            correct_test += (pred==labels).sum().item()
            total_test += labels.size(0)
            all_imgs.append(imgs.cpu())
            all_preds.append(pred.cpu())
            all_labels.append(labels.cpu())
    test_acc = correct_test/total_test
    print(f'Test Accuracy: {test_acc*100:.2f}%')
    wandb.log({'test_accuracy': test_acc})

    # 10x3 Test Grid
    all_imgs = torch.cat(all_imgs)
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    num_images = min(30, len(all_imgs))
    fig, axes = plt.subplots(10,3, figsize=(10,30))
    axes = axes.flatten()
    for i in range(num_images):
        img = all_imgs[i].permute(1,2,0).numpy()
        axes[i].imshow(img)
        axes[i].set_title(f"Pred: {CLASSES[all_preds[i]]}\nTrue: {CLASSES[all_labels[i]]}")
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()
    wandb.log({"test_samples": wandb.Image(fig)})

    # Optional: First-layer filters
    def visualize_first_layer_filters(model):
        first_conv = None
        for layer in model.layers:
            if isinstance(layer, nn.Conv2d):
                first_conv = layer
                break
        if first_conv is None: return
        weights = first_conv.weight.data.cpu()
        num_filters = weights.shape[0]
        fig, axes = plt.subplots(4,8,figsize=(12,6))
        axes = axes.flatten()
        for i in range(min(num_filters,len(axes))):
            filt = weights[i].permute(1,2,0)
            filt = (filt - filt.min())/(filt.max()-filt.min())
            axes[i].imshow(filt)
            axes[i].axis('off')
        plt.suptitle("First Layer Filters")
        plt.show()
        wandb.log({"first_layer_filters": wandb.Image(fig)})
    visualize_first_layer_filters(model)

    # Optional: Guided Backprop
    class GuidedBackprop:
        def __init__(self, model):
            self.model = model
            self.model.eval()
            def backward_hook(module, grad_in, grad_out):
                return (torch.clamp(grad_in[0], min=0.0),)
            for module in self.model.modules():
                if isinstance(module, nn.ReLU):
                    module.register_backward_hook(backward_hook)
        def generate_gradients(self, input_image, target_class):
            input_image = input_image.unsqueeze(0).to(device)
            input_image.requires_grad=True
            output = self.model(input_image)
            self.model.zero_grad()
            loss = output[0,target_class]
            loss.backward()
            gradients = input_image.grad.data.cpu().squeeze()
            gradients = (gradients - gradients.min())/(gradients.max()-gradients.min())
            return gradients.permute(1,2,0).numpy()
    
    gbp = GuidedBackprop(model)
    sample_img, _ = test_data[0]
    fig, axes = plt.subplots(2,5,figsize=(15,6))
    axes = axes.flatten()
    for i in range(10):
        target_class = i % len(CLASSES)
        grad_img = gbp.generate_gradients(sample_img, target_class)
        axes[i].imshow(grad_img)
        axes[i].set_title(f"Neuron {i} -> Class {CLASSES[target_class]}")
        axes[i].axis('off')
    plt.suptitle("Guided Backpropagation on 10 Neurons (CONV5)")
    plt.show()
    wandb.log({"guided_backprop": wandb.Image(fig)})


In [6]:
sweep_cfg = {
    'method':'random',
    'metric':{'name':'val_acc','goal':'maximize'},
    'parameters':{
        'filters':{'values':[[32,64,128,128,256],[64,128,256,256,512]]},
        'activation':{'values':['relu','gelu','silu','mish']},
        'dropout':{'values':[0.2,0.3]},
        'batch_norm':{'values':[True,False]},
        'dense_neurons':{'values':[256,512,1024]},
        'lr':{'values':[0.001,0.0005]}
    }
}

sweep_id = wandb.sweep(sweep_cfg, project=project, entity=entity)
print(f"Sweep created: {sweep_id}")
wandb.agent(sweep_id, train_sweep, count=20)


Create sweep with ID: yd1fqesu
Sweep URL: https://wandb.ai/cs24s023-iitm-ac-in/garbage_classification/sweeps/yd1fqesu
Sweep created: yd1fqesu


wandb: Agent Starting Run: bt6r5dle with config:
wandb: 	activation: mish
wandb: 	batch_norm: True
wandb: 	dense_neurons: 512
wandb: 	dropout: 0.3
wandb: 	filters: [64, 128, 256, 256, 512]
wandb: 	lr: 0.001
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


Traceback (most recent call last):
  File "/home/nagaraj/miniconda3/envs/TrajMIA/lib/python3.11/site-packages/wandb/agents/pyagent.py", line 297, in _run_job
    self._function()
  File "/tmp/ipykernel_1679281/3863473439.py", line 6, in train_sweep
    dense_units=cfg.dense_neurons).to(device)
                                      ^^^^^^
NameError: name 'device' is not defined

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: ERROR Run bt6r5dle errored: name 'device' is not defined
wandb: Agent Starting Run: 02cx2y2g with config:
wandb: 	activation: gelu
wandb: 	batch_norm: True
wandb: 	dense_neurons: 512
wandb: 	dropout: 0.3
wandb: 	filters: [64, 128, 256, 256, 512]
wandb: 	lr: 0.0005
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


Traceback (most recent call last):
  File "/home/nagaraj/miniconda3/envs/TrajMIA/lib/python3.11/site-packages/wandb/agents/pyagent.py", line 297, in _run_job
    self._function()
  File "/tmp/ipykernel_1679281/3863473439.py", line 6, in train_sweep
    dense_units=cfg.dense_neurons).to(device)
                                      ^^^^^^
NameError: name 'device' is not defined

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: ERROR Run 02cx2y2g errored: name 'device' is not defined
wandb: Agent Starting Run: wcsig1ad with config:
wandb: 	activation: relu
wandb: 	batch_norm: False
wandb: 	dense_neurons: 256
wandb: 	dropout: 0.2
wandb: 	filters: [32, 64, 128, 128, 256]
wandb: 	lr: 0.0005
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


Traceback (most recent call last):
  File "/home/nagaraj/miniconda3/envs/TrajMIA/lib/python3.11/site-packages/wandb/agents/pyagent.py", line 297, in _run_job
    self._function()
  File "/tmp/ipykernel_1679281/3863473439.py", line 6, in train_sweep
    dense_units=cfg.dense_neurons).to(device)
                                      ^^^^^^
NameError: name 'device' is not defined

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: ERROR Run wcsig1ad errored: name 'device' is not defined
wandb: ERROR Detected 3 failed runs in the first 60 seconds, killing sweep.
wandb: To disable this check set WANDB_AGENT_DISABLE_FLAPPING=true
